In [ ]:
# # tox21_bilstm.py  ──────────────────────────────────────────────────────────
# import os, numpy as np, pandas as pd, tensorflow as tf
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import (roc_auc_score, accuracy_score,
#                              f1_score, precision_score, recall_score)
# # 1. Load the data ---------------------------------------------------------------
# df = pd.read_csv("tox21.csv")                     
# targets = ['NR-AR','NR-AR-LBD','NR-AhR','NR-Aromatase','NR-ER',
#            'NR-ER-LBD','NR-PPAR-gamma','SR-ARE','SR-ATAD5',
#            'SR-HSE','SR-MMP','SR-p53']
# smiles = df["smiles"].astype(str).tolist()
# y      = df[targets]
# mask   = y.isna()                 # True=loss data
# y_filled = y.fillna(0).values.astype("float32")


# # 2. Character tokeniser -------------------------------------------------------
# tok = tf.keras.preprocessing.text.Tokenizer(char_level=True)
# tok.fit_on_texts(smiles)
# sequences = tok.texts_to_sequences(smiles)
# MAXLEN = 150                               # Cover 95% length of SMILES
# X = tf.keras.preprocessing.sequence.pad_sequences(
#         sequences, maxlen=MAXLEN, padding="post", truncating="post")

# # 3. Split data --------------------------------------------------------------
# X_tr, X_tmp, y_tr, y_tmp, m_tr, m_tmp = train_test_split(
#         X, y_filled, mask.values, test_size=0.30, random_state=42)
# X_va, X_te, y_va, y_te, m_va, m_te = train_test_split(
#         X_tmp, y_tmp, m_tmp, test_size=0.50, random_state=42)

# # 4. Build BiLSTM ------------------------------------------------------
# def build_bilstm(vocab_size, emb_dim=128, hid=32, dropout=0.3, lr=1e-3):
#     inp = tf.keras.Input(shape=(MAXLEN,), dtype="int32")
#     x   = tf.keras.layers.Embedding(vocab_size, emb_dim, mask_zero=True)(inp)
#     x   = tf.keras.layers.Bidirectional(
#             tf.keras.layers.LSTM(hid, dropout=dropout))(x)
#     out = tf.keras.layers.Dense(len(targets), activation="sigmoid")(x)
#     model = tf.keras.Model(inp, out)
#     model.compile(
#         optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
#         loss="binary_crossentropy")
#     return model

# model = build_bilstm(len(tok.word_index)+1,
#                      emb_dim=128, hid=32, dropout=0.3, lr=1e-3)

# # 5. Trainning the model（mask missing label） --------------------------------------------------
# class MaskedBCE(tf.keras.losses.Loss):
#     def call(self, y_true, y_pred):
#         mask = tf.cast(tf.not_equal(y_true, -1.0), tf.float32)
#         loss = tf.keras.backend.binary_crossentropy(y_true, y_pred)
#         return tf.reduce_sum(loss * mask) / tf.reduce_sum(mask)

# model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
#               loss=MaskedBCE())

# early = tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)
# model.fit(X_tr, y_tr, epochs=5, batch_size=128,
#           validation_data=(X_va, y_va), callbacks=[early], verbose=2)

# # 6. Evaluation index --------------------------------------------------------------
# y_prob = model.predict(X_te, batch_size=256)
# metrics = []
# for i, col in enumerate(targets):
#     valid = ~m_te[:, i]
#     if valid.sum() == 0: continue
#     yt, yp = y_te[valid, i], y_prob[valid, i]
#     y_bin  = (yp >= 0.5).astype(int)
#     auc = roc_auc_score(yt, yp) if len(np.unique(yt))==2 else np.nan
#     acc = accuracy_score(yt, y_bin)
#     f1  = f1_score(yt, y_bin, zero_division=0)
#     pre = precision_score(yt, y_bin, zero_division=0)
#     rec = recall_score(yt, y_bin, zero_division=0)
#     metrics.append([col, auc, acc, f1, pre, rec])

# df_metrics = pd.DataFrame(metrics,
#              columns=["Target","AUC","Accuracy","F1","Precision","Recall"])
# # df_metrics.to_csv("bilstm_tox21_metrics.csv", index=False)
# print(df_metrics)
# # ───────────────────────────────────────────────────────────────────────────


In [ ]:
# # tox21_bilstm_weighted.py
# # ──────────────────────────────────────────────────────────────────────────
# import os, random
# import numpy as np, pandas as pd, tensorflow as tf
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score,
#                              precision_score, recall_score)
# from tqdm import tqdm

# SEED = 42
# random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# # 1. Load the data ────────────────────────────────────────────────────────────
# CSV_PATH = "tox21.csv"                   
# df = pd.read_csv(CSV_PATH)

# targets = ['NR-AR','NR-AR-LBD','NR-AhR','NR-Aromatase','NR-ER',
#            'NR-ER-LBD','NR-PPAR-gamma','SR-ARE','SR-ATAD5',
#            'SR-HSE','SR-MMP','SR-p53']

# # label use -1 as sentinel；for mask loss data later
# y_sen = df[targets].fillna(-1).astype('float32').values
# smiles = df["smiles"].astype(str).tolist()

# # 2. Character Tokeniser & PAD ──────────────────────────────────────────────
# tok = tf.keras.preprocessing.text.Tokenizer(char_level=True)
# tok.fit_on_texts(smiles)
# seqs = tok.texts_to_sequences(smiles)

# MAXLEN = 150
# X = tf.keras.preprocessing.sequence.pad_sequences(
#         seqs, maxlen=MAXLEN, padding="post", truncating="post")

# # 3. Split the data ───────────────────────────────────────────────────
# X_tr, X_tmp, y_tr, y_tmp = train_test_split(
#         X, y_sen, test_size=0.30, random_state=SEED)
# X_va, X_te, y_va, y_te = train_test_split(
#         X_tmp, y_tmp, test_size=0.50, random_state=SEED)

# # 4. Calculate pos_weight  (only use training set) ────────────────────────────────────
# pos = (y_tr == 1).sum(axis=0).astype('float32')
# neg = (y_tr == 0).sum(axis=0).astype('float32')
# eps = 1e-6
# pos_weight = (neg / (pos + eps))           # shape (12,)

# # 5. Build Bi-LSTM ───────────────────────────────────────────────────
# VOCAB = len(tok.word_index)+1
# def build_model(vocab_size, emb_dim=128, hid=32, dropout=0.3):
#     inp = tf.keras.Input(shape=(MAXLEN,), dtype="int32")
#     x   = tf.keras.layers.Embedding(vocab_size, emb_dim,
#                                     mask_zero=True)(inp)
#     x   = tf.keras.layers.Bidirectional(
#             tf.keras.layers.LSTM(hid, dropout=dropout))(x)
#     out = tf.keras.layers.Dense(len(targets), activation="sigmoid")(x)
#     return tf.keras.Model(inp, out)

# model = build_model(VOCAB)

# # 6. Define Masked + Weighted BCE ────────────────────────────────────────
# pos_w_const = tf.constant(pos_weight, dtype=tf.float32)

# def masked_weighted_bce(y_true, y_pred):
#     """
#     y_true:  -1 = missing, 0/1 = valid label
#     """
#     valid = tf.not_equal(y_true, -1.0)                 # bool mask
#     y_clean = tf.where(valid, y_true, 0.)              # occupied 0
#     # element level BCE
#     l = tf.keras.backend.binary_crossentropy(y_clean, y_pred)
#     # Add category weight: Only applies to positive samples (y_clean==1)
#     weights = tf.where(tf.equal(y_clean, 1.0),
#                        pos_w_const, tf.ones_like(pos_w_const))
#     l = l * weights
#     # Take the average only at the valid position
#     l = tf.reduce_sum(tf.where(valid, l, 0.)) \
#         / tf.reduce_sum(tf.cast(valid, tf.float32))
#     return l

# model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
#               loss=masked_weighted_bce)

# # 7. Trainning the model ────────────────────────────────────────────────────────────────
# early = tf.keras.callbacks.EarlyStopping(patience=3,
#                                          restore_best_weights=True)
# model.fit(X_tr, y_tr, epochs=10, batch_size=128,
#           validation_data=(X_va, y_va), callbacks=[early], verbose=2)

# # 8. Evaluation index ────────────────────────────────────────────────────────────
# y_prob = model.predict(X_te, batch_size=256, verbose=0)

# results = []
# for i, col in enumerate(targets):
#     valid = y_te[:, i] != -1
#     if valid.sum() == 0:
#         continue
#     yt, yp = y_te[valid, i], y_prob[valid, i]
#     y_bin  = (yp >= 0.5).astype(int)

#     auc = roc_auc_score(yt, yp) if len(np.unique(yt)) == 2 else np.nan
#     acc = accuracy_score(yt, y_bin)
#     f1  = f1_score(yt, y_bin, zero_division=0)
#     pre = precision_score(yt, y_bin, zero_division=0)
#     rec = recall_score(yt, y_bin, zero_division=0)
#     results.append([col, auc, acc, f1, pre, rec])
    

# df_metrics = pd.DataFrame(results,
#         columns=["Target","AUC","Accuracy","F1","Precision","Recall"])
# stats_cols = ["AUC","Accuracy","F1","Precision","Recall"]
# mean_row   = df_metrics[stats_cols].mean(skipna=True).to_frame().T
# median_row = df_metrics[stats_cols].median(skipna=True).to_frame().T

# mean_row.insert(0, "Target", "Overall Mean")
# median_row.insert(0, "Target", "Overall Median")

# df_metrics = pd.concat([df_metrics, mean_row, median_row], ignore_index=True)
# # df_metrics.to_csv("BiLSTM_Tox21_metrics_weighted.csv", index=False)
# print("\n=== Test-set metrics ===")
# # print(df_metrics.to_string(index=False))
# print(df_metrics)
# # ──────────────────────────────────────────────────────────────────────────


Epoch 1/10
44/44 - 8s - 180ms/step - loss: 1.2699 - val_loss: 1.2560
Epoch 2/10
44/44 - 7s - 151ms/step - loss: 1.2116 - val_loss: 1.2256
Epoch 3/10
44/44 - 8s - 186ms/step - loss: 1.1662 - val_loss: 1.1658
Epoch 4/10
44/44 - 8s - 176ms/step - loss: 1.1242 - val_loss: 1.1597
Epoch 5/10
44/44 - 8s - 185ms/step - loss: 1.1036 - val_loss: 1.1462
Epoch 6/10
44/44 - 10s - 238ms/step - loss: 1.1012 - val_loss: 1.1326
Epoch 7/10
44/44 - 16s - 354ms/step - loss: 1.0843 - val_loss: 1.1426
Epoch 8/10
44/44 - 13s - 294ms/step - loss: 1.0949 - val_loss: 1.1128
Epoch 9/10
44/44 - 12s - 272ms/step - loss: 1.0913 - val_loss: 1.1326
Epoch 10/10
44/44 - 9s - 214ms/step - loss: 1.0920 - val_loss: 1.1421

=== Test-set metrics ===
            Target       AUC  Accuracy        F1  Precision    Recall
0            NR-AR  0.790932  0.932624  0.387097   0.292683  0.571429
1        NR-AR-LBD  0.771321  0.932187  0.323810   0.226667  0.566667
2           NR-AhR  0.784760  0.698113  0.355932   0.233983  0.743363

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
import os, random
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score,
                             precision_score, recall_score)
from tqdm import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# 1. Load the data ────────────────────────────────────────────────────────────
CSV_PATH = "tox21.csv"                     
df = pd.read_csv(CSV_PATH)

targets = ['NR-AR','NR-AR-LBD','NR-AhR','NR-Aromatase','NR-ER',
           'NR-ER-LBD','NR-PPAR-gamma','SR-ARE','SR-ATAD5',
           'SR-HSE','SR-MMP','SR-p53']

# Label use -1 as sentinel, for mask loss data later
y_sen = df[targets].fillna(-1).astype('float32').values
smiles = df["smiles"].astype(str).tolist()

# 2. Character level Tokeniser & PAD ──────────────────────────────────────────────
tok = tf.keras.preprocessing.text.Tokenizer(char_level=True)
tok.fit_on_texts(smiles)
seqs = tok.texts_to_sequences(smiles)

MAXLEN = 150
X = tf.keras.preprocessing.sequence.pad_sequences(
        seqs, maxlen=MAXLEN, padding="post", truncating="post")

# 3. Split the data ───────────────────────────────────────────────────
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        X, y_sen, test_size=0.30, random_state=SEED)
X_va, X_te, y_va, y_te = train_test_split(
        X_tmp, y_tmp, test_size=0.50, random_state=SEED)

# 4. Calculate  pos_weight  (Only use the training set for statistics) ────────────────────────────────────
pos = (y_tr == 1).sum(axis=0).astype('float32')
neg = (y_tr == 0).sum(axis=0).astype('float32')
eps = 1e-6
pos_weight = (neg / (pos + eps))           # shape (12,)

# 5. Build Bi-LSTM model ───────────────────────────────────────────────────
VOCAB = len(tok.word_index)+1
def build_model(vocab_size, emb_dim=256, hid=64, dropout=0.3):
    inp = tf.keras.Input(shape=(MAXLEN,), dtype="int32")
    x   = tf.keras.layers.Embedding(vocab_size, emb_dim,
                                    mask_zero=True)(inp)
    x   = tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(hid, dropout=dropout))(x)
    out = tf.keras.layers.Dense(len(targets), activation="sigmoid")(x)
    return tf.keras.Model(inp, out)

model = build_model(VOCAB)

# 6. Define Masked + Weighted BCE ────────────────────────────────────────
pos_w_const = tf.constant(pos_weight, dtype=tf.float32)

def masked_weighted_bce(y_true, y_pred):
    """
    y_true:  -1 = missing, 0/1 = valid label
    """
    valid = tf.not_equal(y_true, -1.0)                 # bool mask
    y_clean = tf.where(valid, y_true, 0.)              # 占位 0
    # Element level BCE
    l = tf.keras.backend.binary_crossentropy(y_clean, y_pred)
    # Add category weight: Only applies to positive samples  (y_clean==1)
    weights = tf.where(tf.equal(y_clean, 1.0),
                       pos_w_const, tf.ones_like(pos_w_const))
    l = l * weights
    # Take the average only at the valid position
    l = tf.reduce_sum(tf.where(valid, l, 0.)) \
        / tf.reduce_sum(tf.cast(valid, tf.float32))
    return l

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss=masked_weighted_bce)

# 7. Fit the model ────────────────────────────────────────────────────────────────
early = tf.keras.callbacks.EarlyStopping(patience=3,
                                         restore_best_weights=True)
model.fit(X_tr, y_tr, epochs=10, batch_size=128,
          validation_data=(X_va, y_va), callbacks=[early], verbose=2)

# 8. Evaluation index ────────────────────────────────────────────────────────────
y_prob = model.predict(X_te, batch_size=256, verbose=0)

results = []
for i, col in enumerate(targets):
    valid = y_te[:, i] != -1
    if valid.sum() == 0:
        continue
    yt, yp = y_te[valid, i], y_prob[valid, i]
    y_bin  = (yp >= 0.5).astype(int)

    auc = roc_auc_score(yt, yp) if len(np.unique(yt)) == 2 else np.nan
    acc = accuracy_score(yt, y_bin)
    f1  = f1_score(yt, y_bin, zero_division=0)
    pre = precision_score(yt, y_bin, zero_division=0)
    rec = recall_score(yt, y_bin, zero_division=0)
    results.append([col, auc, acc, f1, pre, rec])
    

df_metrics = pd.DataFrame(results,
        columns=["Target","AUC","Accuracy","F1","Precision","Recall"])
stats_cols = ["AUC","Accuracy","F1","Precision","Recall"]
mean_row   = df_metrics[stats_cols].mean(skipna=True).to_frame().T
median_row = df_metrics[stats_cols].median(skipna=True).to_frame().T

mean_row.insert(0, "Target", "Overall Mean")
median_row.insert(0, "Target", "Overall Median")

df_metrics = pd.concat([df_metrics, mean_row, median_row], ignore_index=True)
# df_metrics.to_csv("BiLSTM_Tox21_metrics_weighted.csv", index=False)
print("\n=== Test-set metrics ===")
# print(df_metrics.to_string(index=False))
print(df_metrics)
# ──────────────────────────────────────────────────────────────────────────

Epoch 1/10
44/44 - 12s - 266ms/step - loss: 1.2494 - val_loss: 1.2234
Epoch 2/10
44/44 - 10s - 236ms/step - loss: 1.1665 - val_loss: 1.1559
Epoch 3/10
44/44 - 12s - 273ms/step - loss: 1.1162 - val_loss: 1.1624
Epoch 4/10
44/44 - 12s - 273ms/step - loss: 1.0985 - val_loss: 1.1483
Epoch 5/10
44/44 - 12s - 283ms/step - loss: 1.0675 - val_loss: 1.1098
Epoch 6/10
44/44 - 12s - 265ms/step - loss: 1.0663 - val_loss: 1.1272
Epoch 7/10
44/44 - 12s - 279ms/step - loss: 1.0537 - val_loss: 1.1178
Epoch 8/10
44/44 - 14s - 309ms/step - loss: 1.0300 - val_loss: 1.1184

=== Test-set metrics ===
            Target       AUC  Accuracy        F1  Precision    Recall
0            NR-AR  0.810598  0.901596  0.310559   0.210084  0.595238
1        NR-AR-LBD  0.791740  0.920726  0.290598   0.195402  0.566667
2           NR-AhR  0.794401  0.724926  0.374718   0.251515  0.734513
3     NR-Aromatase  0.763975  0.727884  0.187291   0.108949  0.666667
4            NR-ER  0.686131  0.606383  0.312268   0.202899  0.6

In [ ]:
# tox21_bilstm_randsmiles_attn_focal.py
# ────────────────────────────────────────────────────────────
import os, random, numpy as np, pandas as pd, tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score,
                             precision_score, recall_score)
from rdkit import Chem
from tqdm.auto import tqdm     

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ========== 0. Data path & Task column name ============================================
CSV_PATH = "tox21.csv"                       
targets = ['NR-AR','NR-AR-LBD','NR-AhR','NR-Aromatase','NR-ER',
           'NR-ER-LBD','NR-PPAR-gamma','SR-ARE','SR-ATAD5',
           'SR-HSE','SR-MMP','SR-p53']

# ========== 1. Load the data ==============================================
df = pd.read_csv(CSV_PATH)
base_smiles = df["smiles"].astype(str).tolist()
y_base      = df[targets].fillna(-1).astype('float32').values  # -1 sentinel

# ========== 2. Random-SMILES data augmentation ====================================
AUG_FACTOR = 3          # The number of additional random SMILES for each molecule can be adjusted from 0 to 5
aug_smiles, aug_labels = [], []

if AUG_FACTOR > 0:
    print(f"⏫ Augmenting with Random-SMILES ×{AUG_FACTOR} …")
for s, y in tqdm(list(zip(base_smiles, y_base)), total=len(base_smiles)):
    aug_smiles.append(s); aug_labels.append(y)
    if AUG_FACTOR == 0: continue
    mol = Chem.MolFromSmiles(s)
    if mol is None:           # Skip the augmentation if it cannot be parsed
        continue
    for _ in range(AUG_FACTOR):
        rs = Chem.MolToSmiles(mol, doRandom=True)
        aug_smiles.append(rs); aug_labels.append(y)

smiles = aug_smiles
y_all  = np.vstack(aug_labels)

print(f"Dataset size after augmentation: {len(smiles)} SMILES")

# ========== 3. Character level Tokenizer & Padding ================================
tok = tf.keras.preprocessing.text.Tokenizer(char_level=True)
tok.fit_on_texts(smiles)
seqs = tok.texts_to_sequences(smiles)

MAXLEN = 150
X = tf.keras.preprocessing.sequence.pad_sequences(
        seqs, maxlen=MAXLEN, padding="post", truncating="post")

VOCAB = len(tok.word_index) + 1

# ========== 4. Split the data with seed =====================
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        X, y_all, test_size=0.30, random_state=SEED, shuffle=True)
X_va, X_te, y_va, y_te = train_test_split(
        X_tmp, y_tmp, test_size=0.50, random_state=SEED, shuffle=True)

# ========== 5. Two-Layer BiLSTM + Masked Self-Attention ==================
class MaskedAttention(tf.keras.layers.Layer):
    def __init__(self): super().__init__()
    def build(self, input_shape):
        self.score = tf.keras.layers.Dense(1, activation='tanh')
    def call(self, x, mask=None):          # x:[B,T,H]
        e = tf.squeeze(self.score(x), axis=-1)       # [B,T]
        if mask is not None:
            e -= 1e9 * (1 - tf.cast(mask, tf.float32))   # mask pad
        w = tf.nn.softmax(e, axis=1)                    # attention weights
        out = tf.reduce_sum(x * tf.expand_dims(w, -1), axis=1) # [B,H]
        return out

def build_model(vocab, emb_dim=128, hid=64, dropout=0.3):
    inp = tf.keras.Input(shape=(MAXLEN,), dtype='int32')
    emb = tf.keras.layers.Embedding(vocab, emb_dim, mask_zero=True)(inp)
    x = tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(hid, return_sequences=True,
                                 dropout=dropout))(emb)
    x = tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(hid, return_sequences=True,
                                 dropout=dropout))(x)
    att = MaskedAttention()(x)
    out = tf.keras.layers.Dense(len(targets), activation='sigmoid')(att)
    return tf.keras.Model(inp, out)

model = build_model(VOCAB)

# ========== 6. Masked Focal Loss (γ=2, α=0.25) ===========================
def masked_focal(y_true, y_pred, γ=2.0, α=0.25):
    valid = tf.not_equal(y_true, -1.0)
    y_clean = tf.where(valid, y_true, 0.)
    pt = y_clean * y_pred + (1 - y_clean) * (1 - y_pred)
    loss = -α * tf.pow(1 - pt, γ) * tf.math.log(tf.clip_by_value(pt, 1e-7, 1.0))
    loss = tf.where(valid, loss, 0.)
    return tf.reduce_sum(loss) / tf.reduce_sum(tf.cast(valid, tf.float32))

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=5.0),
              loss=masked_focal)

early = tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)

# ========== 7. Fit the model =======================================================
model.fit(X_tr, y_tr, epochs=30, batch_size=128,
          validation_data=(X_va, y_va), callbacks=[early], verbose=2)

# ========== 8. Search for the optimal threshold of each task based on the validation set ===============================
val_prob = model.predict(X_va, batch_size=256, verbose=0)
mask_va = (y_va == -1)
best_thr = []
for i in range(len(targets)):
    v = ~mask_va[:, i]
    if v.sum() == 0:
        best_thr.append(0.5); continue
    yt, yp = y_va[v, i], val_prob[v, i]
    ts = np.linspace(0.1, 0.9, 17)
    f1s = [f1_score(yt, yp >= t, zero_division=0) for t in ts]
    best_thr.append(ts[int(np.argmax(f1s))])

# ========== 9. Test Set Evaluation ======================================
test_prob = model.predict(X_te, batch_size=256, verbose=0)
mask_te   = (y_te == -1)

rows = []
for i, col in enumerate(targets):
    v = ~mask_te[:, i]
    if v.sum() == 0: continue
    yt, yp = y_te[v, i], test_prob[v, i]
    y_bin = (yp >= best_thr[i]).astype(int)
    auc = roc_auc_score(yt, yp) if len(np.unique(yt))==2 else np.nan
    acc = accuracy_score(yt, y_bin)
    f1  = f1_score(yt, y_bin, zero_division=0)
    pre = precision_score(yt, y_bin, zero_division=0)
    rec = recall_score(yt, y_bin, zero_division=0)
    rows.append([col, auc, acc, f1, pre, rec])

df = pd.DataFrame(rows, columns=["Target","AUC","Accuracy","F1","Precision","Recall"])
# overall mean / median
stats = ["AUC","Accuracy","F1","Precision","Recall"]
df_mean   = df[stats].mean().to_frame().T
df_median = df[stats].median().to_frame().T
df_mean.insert(0,"Target","Overall Mean")
df_median.insert(0,"Target","Overall Median")
df = pd.concat([df, df_mean, df_median], ignore_index=True)

# ========== 10. 保存 & 打印 ==============================================
#OUT_CSV = "BiLSTM_Tox21_rand_attn_focal_metrics.csv"
#df.to_csv(OUT_CSV, index=False)
print(f"\n=== Test-set metrics ===")
print(df)


c:\Users\LJM\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⏫ Augmenting with Random-SMILES ×3 …


100%|██████████| 8006/8006 [00:02<00:00, 2781.97it/s]


Dataset size after augmentation: 32024 SMILES

Epoch 1/30


c:\Users\LJM\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\layer.py:932: UserWarning: Layer 'masked_attention' (of type MaskedAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


176/176 - 221s - 1s/step - loss: 0.0175 - val_loss: 0.0160
Epoch 2/30
176/176 - 221s - 1s/step - loss: 0.0157 - val_loss: 0.0153
Epoch 3/30
176/176 - 236s - 1s/step - loss: 0.0153 - val_loss: 0.0151
Epoch 4/30
176/176 - 253s - 1s/step - loss: 0.0151 - val_loss: 0.0149
Epoch 5/30
176/176 - 232s - 1s/step - loss: 0.0148 - val_loss: 0.0148
Epoch 6/30
176/176 - 249s - 1s/step - loss: 0.0146 - val_loss: 0.0147
Epoch 7/30
176/176 - 258s - 1s/step - loss: 0.0145 - val_loss: 0.0145
Epoch 8/30
176/176 - 354s - 2s/step - loss: 0.0142 - val_loss: 0.0143
Epoch 9/30
176/176 - 327s - 2s/step - loss: 0.0140 - val_loss: 0.0141
Epoch 10/30
176/176 - 367s - 2s/step - loss: 0.0138 - val_loss: 0.0139
Epoch 11/30
176/176 - 316s - 2s/step - loss: 0.0135 - val_loss: 0.0137
Epoch 12/30
176/176 - 370s - 2s/step - loss: 0.0133 - val_loss: 0.0135
Epoch 13/30
176/176 - 308s - 2s/step - loss: 0.0130 - val_loss: 0.0133
Epoch 14/30
176/176 - 268s - 2s/step - loss: 0.0128 - val_loss: 0.0132
Epoch 15/30
176/176 - 283s

NameError: name 'OUT_CSV' is not defined

In [6]:
print(f"\n=== Test-set metrics ===")
print(df)


=== Test-set metrics ===
            Target       AUC  Accuracy        F1  Precision    Recall
0            NR-AR  0.861643  0.975118  0.636066   0.850877  0.507853
1        NR-AR-LBD  0.917410  0.974186  0.644518   0.673611  0.617834
2           NR-AhR  0.892682  0.889917  0.526882   0.542035  0.512552
3     NR-Aromatase  0.882005  0.907017  0.409420   0.333333  0.530516
4            NR-ER  0.767364  0.869577  0.476907   0.476395  0.477419
5        NR-ER-LBD  0.889392  0.955493  0.527363   0.579235  0.484018
6    NR-PPAR-gamma  0.877563  0.965526  0.383562   0.392523  0.375000
7           SR-ARE  0.831420  0.826404  0.525346   0.513514  0.537736
8         SR-ATAD5  0.893458  0.934558  0.379085   0.291946  0.540373
9           SR-HSE  0.860756  0.925369  0.439771   0.402098  0.485232
10          SR-MMP  0.900161  0.856336  0.610559   0.554937  0.678571
11          SR-p53  0.876681  0.905583  0.379585   0.321622  0.463035
12    Overall Mean  0.870878  0.915424  0.494922   0.494344  0.5

### PyTorch BiLSTM

In [8]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Load data
df = pd.read_csv("tox21.csv")
target_cols = [
    'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER',
    'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5',
    'SR-HSE', 'SR-MMP', 'SR-p53'
]
df = df.dropna(subset=target_cols).reset_index(drop=True)
smiles = df['smiles'].values
labels = df[target_cols].astype(int).values

# Train-test split
X_train, X_temp, y_train, y_temp = train_test_split(smiles, labels, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Tokenizer
class CharTokenizer:
    def __init__(self, smiles_list):
        chars = sorted(set(''.join(smiles_list)))
        self.char2idx = {ch: i + 1 for i, ch in enumerate(chars)}
        self.idx2char = {i: ch for ch, i in self.char2idx.items()}
        self.vocab_size = len(self.char2idx) + 1

    def encode(self, smiles, max_len=120):
        encoded = [self.char2idx.get(ch, 0) for ch in smiles[:max_len]]
        return encoded + [0] * (max_len - len(encoded))

tokenizer = CharTokenizer(X_train)
max_len = 120

class Tox21Dataset(Dataset):
    def __init__(self, smiles_list, labels, tokenizer, max_len):
        self.inputs = [tokenizer.encode(smiles, max_len) for smiles in smiles_list]
        self.labels = labels

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return torch.tensor(self.inputs[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.float)

train_loader = DataLoader(Tox21Dataset(X_train, y_train, tokenizer, max_len), batch_size=64, shuffle=True)
val_loader = DataLoader(Tox21Dataset(X_val, y_val, tokenizer, max_len), batch_size=64)
test_loader = DataLoader(Tox21Dataset(X_test, y_test, tokenizer, max_len), batch_size=64)

# Model
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        hidden = torch.cat((hidden[0], hidden[1]), dim=1)
        return self.fc(hidden)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BiLSTMClassifier(tokenizer.vocab_size, embed_dim=128, hidden_dim=64, output_dim=12).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Training loop
def train(model, loader):
    model.train()
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

# Evaluation
def evaluate(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x).cpu()
            y_pred.append(torch.sigmoid(logits))
            y_true.append(y)
    return torch.cat(y_true).numpy(), torch.cat(y_pred).numpy()

for epoch in range(10):
    train(model, train_loader)

y_true, y_score = evaluate(model, test_loader)
y_pred = (y_score >= 0.5).astype(int)

# Metrics
metrics = {
    'Label': [],
    'AUC': [], 'Accuracy': [], 'F1': [], 'Precision': [], 'Recall': []
}

for i, label in enumerate(target_cols):
    try:
        auc = roc_auc_score(y_true[:, i], y_score[:, i])
    except:
        auc = np.nan
    metrics['Label'].append(label)
    metrics['AUC'].append(auc)
    metrics['Accuracy'].append(accuracy_score(y_true[:, i], y_pred[:, i]))
    metrics['F1'].append(f1_score(y_true[:, i], y_pred[:, i]))
    metrics['Precision'].append(precision_score(y_true[:, i], y_pred[:, i]))
    metrics['Recall'].append(recall_score(y_true[:, i], y_pred[:, i]))

df_metrics = pd.DataFrame(metrics)
df_metrics.loc['mean'] = df_metrics.mean(numeric_only=True)
df_metrics.loc['median'] = df_metrics.median(numeric_only=True)
print(df_metrics)


c:\Users\LJM\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\LJM\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\LJM\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()

                Label       AUC  Accuracy   F1  Precision  Recall
0               NR-AR  0.725094  0.977492  0.0        0.0     0.0
1           NR-AR-LBD  0.775163  0.983923  0.0        0.0     0.0
2              NR-AhR  0.753814  0.948553  0.0        0.0     0.0
3        NR-Aromatase  0.714521  0.974277  0.0        0.0     0.0
4               NR-ER  0.486305  0.906752  0.0        0.0     0.0
5           NR-ER-LBD  0.539474  0.977492  0.0        0.0     0.0
6       NR-PPAR-gamma  0.850649  0.990354  0.0        0.0     0.0
7              SR-ARE  0.674838  0.938907  0.0        0.0     0.0
8            SR-ATAD5       NaN  1.000000  0.0        0.0     0.0
9              SR-HSE  0.307756  0.974277  0.0        0.0     0.0
10             SR-MMP  0.704425  0.954984  0.0        0.0     0.0
11             SR-p53  0.578431  0.983923  0.0        0.0     0.0
mean              NaN  0.646406  0.967578  0.0        0.0     0.0
median            NaN  0.689631  0.974277  0.0        0.0     0.0


c:\Users\LJM\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\LJM\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\LJM\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()

In [10]:
# BiLSTM with pos_weight, validation, dropout, threshold tuning
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Load and clean dataset
df = pd.read_csv("tox21.csv")
target_cols = [
    'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER',
    'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5',
    'SR-HSE', 'SR-MMP', 'SR-p53'
]
df = df.dropna(subset=target_cols).reset_index(drop=True)
smiles = df['smiles'].values
labels = df[target_cols].astype(int).values

# Split
token_max_len = 120
X_train, X_temp, y_train, y_temp = train_test_split(smiles, labels, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Char tokenizer
class CharTokenizer:
    def __init__(self, smiles_list):
        chars = sorted(set(''.join(smiles_list)))
        self.char2idx = {ch: i + 1 for i, ch in enumerate(chars)}
        self.idx2char = {i: ch for ch, i in self.char2idx.items()}
        self.vocab_size = len(self.char2idx) + 1

    def encode(self, smiles, max_len=120):
        encoded = [self.char2idx.get(ch, 0) for ch in smiles[:max_len]]
        return encoded + [0] * (max_len - len(encoded))

tokenizer = CharTokenizer(X_train)

# Dataset class
class Tox21Dataset(Dataset):
    def __init__(self, smiles_list, labels, tokenizer, max_len):
        self.inputs = [tokenizer.encode(smiles, max_len) for smiles in smiles_list]
        self.labels = labels

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return torch.tensor(self.inputs[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.float)

# DataLoaders
train_loader = DataLoader(Tox21Dataset(X_train, y_train, tokenizer, token_max_len), batch_size=64, shuffle=True)
val_loader = DataLoader(Tox21Dataset(X_val, y_val, tokenizer, token_max_len), batch_size=64)
test_loader = DataLoader(Tox21Dataset(X_test, y_test, tokenizer, token_max_len), batch_size=64)

# Pos_weight calculation
pos_weight = torch.tensor((y_train == 0).sum(axis=0) / (y_train == 1).sum(axis=0), dtype=torch.float)

# BiLSTM model with Dropout (ADDED HERE)
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(0.3)  # <-- ADDED DROPOUT
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        hidden = torch.cat((hidden[0], hidden[1]), dim=1)
        x = self.dropout(hidden)  # <-- ADDED DROPOUT
        return self.fc(x)

# Model setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BiLSTMClassifier(tokenizer.vocab_size, 128, 64, 12).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))  # <-- ADDED pos_weight
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Training loop with validation monitoring
best_val_auc = 0
for epoch in range(30):  # <-- INCREASED TO 30 EPOCHS
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

    # Evaluate on validation set
    model.eval()
    y_val_true, y_val_score = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            logits = model(x).cpu()
            y_val_score.append(torch.sigmoid(logits))
            y_val_true.append(y)
    y_val_true = torch.cat(y_val_true).numpy()
    y_val_score = torch.cat(y_val_score).numpy()
    aucs = [roc_auc_score(y_val_true[:, i], y_val_score[:, i]) for i in range(12) if len(np.unique(y_val_true[:, i])) > 1]
    mean_auc = np.mean(aucs)
    print(f"Epoch {epoch + 1}, Validation AUC: {mean_auc:.4f}")

# Predict on test set
def predict(model, loader):
    model.eval()
    y_true, y_score = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x).cpu()
            y_score.append(torch.sigmoid(logits))
            y_true.append(y)
    return torch.cat(y_true).numpy(), torch.cat(y_score).numpy()

y_true, y_score = predict(model, test_loader)

# Tune threshold per class on val set
def tune_threshold(y_true, y_score):
    thresholds = []
    for i in range(y_true.shape[1]):
        best_t, best_f1 = 0.5, 0
        for t in np.arange(0.1, 0.9, 0.05):
            pred = (y_score[:, i] >= t).astype(int)
            score = f1_score(y_true[:, i], pred)
            if score > best_f1:
                best_f1 = score
                best_t = t
        thresholds.append(best_t)
    return thresholds

# Use val set to tune thresholds
y_val_true, y_val_score = predict(model, val_loader)
optimal_thresholds = tune_threshold(y_val_true, y_val_score)

# Apply thresholds to test set
y_pred = np.zeros_like(y_score)
for i in range(12):
    y_pred[:, i] = (y_score[:, i] >= optimal_thresholds[i]).astype(int)

# Metrics
target_cols = [
    'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER',
    'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5',
    'SR-HSE', 'SR-MMP', 'SR-p53'
]
metrics = {'Label': [], 'AUC': [], 'Accuracy': [], 'F1': [], 'Precision': [], 'Recall': []}
for i, label in enumerate(target_cols):
    try:
        auc = roc_auc_score(y_true[:, i], y_score[:, i])
    except:
        auc = np.nan
    metrics['Label'].append(label)
    metrics['AUC'].append(auc)
    metrics['Accuracy'].append(accuracy_score(y_true[:, i], y_pred[:, i]))
    metrics['F1'].append(f1_score(y_true[:, i], y_pred[:, i]))
    metrics['Precision'].append(precision_score(y_true[:, i], y_pred[:, i]))
    metrics['Recall'].append(recall_score(y_true[:, i], y_pred[:, i]))

df_metrics = pd.DataFrame(metrics)
df_metrics.loc['mean'] = df_metrics.mean(numeric_only=True)
df_metrics.loc['median'] = df_metrics.median(numeric_only=True)
print(df_metrics)

Epoch 1, Validation AUC: 0.6052
Epoch 2, Validation AUC: 0.6154
Epoch 3, Validation AUC: 0.6236
Epoch 4, Validation AUC: 0.6372
Epoch 5, Validation AUC: 0.6588
Epoch 6, Validation AUC: 0.6933
Epoch 7, Validation AUC: 0.6967
Epoch 8, Validation AUC: 0.7139
Epoch 9, Validation AUC: 0.7072
Epoch 10, Validation AUC: 0.7346
Epoch 11, Validation AUC: 0.7266
Epoch 12, Validation AUC: 0.7343
Epoch 13, Validation AUC: 0.7218
Epoch 14, Validation AUC: 0.7340
Epoch 15, Validation AUC: 0.7388
Epoch 16, Validation AUC: 0.7570
Epoch 17, Validation AUC: 0.7499
Epoch 18, Validation AUC: 0.7433
Epoch 19, Validation AUC: 0.7526
Epoch 20, Validation AUC: 0.7433
Epoch 21, Validation AUC: 0.7588
Epoch 22, Validation AUC: 0.7591
Epoch 23, Validation AUC: 0.7566
Epoch 24, Validation AUC: 0.7618
Epoch 25, Validation AUC: 0.7576
Epoch 26, Validation AUC: 0.7603
Epoch 27, Validation AUC: 0.7694
Epoch 28, Validation AUC: 0.7575
Epoch 29, Validation AUC: 0.7648
Epoch 30, Validation AUC: 0.7539
                Lab

c:\Users\LJM\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Keras (TensorFlow)

In [9]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

# Load data
df = pd.read_csv("tox21.csv")
target_cols = [
    'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER',
    'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5',
    'SR-HSE', 'SR-MMP', 'SR-p53'
]
df = df.dropna(subset=target_cols).reset_index(drop=True)
smiles = df['smiles'].values
labels = df[target_cols].astype(int).values

# Tokenizer
tokenizer = Tokenizer(char_level=True)
tokenizer.fit_on_texts(smiles)
sequences = tokenizer.texts_to_sequences(smiles)
X = pad_sequences(sequences, maxlen=120, padding='post')
X_train, X_temp, y_train, y_temp = train_test_split(X, labels, test_size=0.2)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5)

# Model
inp = Input(shape=(120,))
x = Embedding(input_dim=len(tokenizer.word_index)+1, output_dim=128)(inp)
x = Bidirectional(LSTM(64))(x)
out = Dense(12, activation='sigmoid')(x)
model = Model(inputs=inp, outputs=out)
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=10, batch_size=64)

# Prediction
y_score = model.predict(X_test)
y_pred = (y_score >= 0.5).astype(int)

# Metrics
metrics = {
    'Label': [],
    'AUC': [], 'Accuracy': [], 'F1': [], 'Precision': [], 'Recall': []
}

for i, label in enumerate(target_cols):
    try:
        auc = roc_auc_score(y_test[:, i], y_score[:, i])
    except:
        auc = np.nan
    metrics['Label'].append(label)
    metrics['AUC'].append(auc)
    metrics['Accuracy'].append(accuracy_score(y_test[:, i], y_pred[:, i]))
    metrics['F1'].append(f1_score(y_test[:, i], y_pred[:, i]))
    metrics['Precision'].append(precision_score(y_test[:, i], y_pred[:, i]))
    metrics['Recall'].append(recall_score(y_test[:, i], y_pred[:, i]))

df_metrics = pd.DataFrame(metrics)
df_metrics.loc['mean'] = df_metrics.mean(numeric_only=True)
df_metrics.loc['median'] = df_metrics.median(numeric_only=True)
print(df_metrics)


Epoch 1/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 10s 137ms/step - accuracy: 0.0200 - loss: 0.3635 - val_accuracy: 0.0579 - val_loss: 0.1414
Epoch 2/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 9s 241ms/step - accuracy: 0.0468 - loss: 0.1107 - val_accuracy: 0.0579 - val_loss: 0.1398
Epoch 3/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 7s 181ms/step - accuracy: 0.0456 - loss: 0.1103 - val_accuracy: 0.0579 - val_loss: 0.1398
Epoch 4/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 7s 186ms/step - accuracy: 0.0464 - loss: 0.1103 - val_accuracy: 0.0579 - val_loss: 0.1399
Epoch 5/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 7s 170ms/step - accuracy: 0.0463 - loss: 0.1102 - val_accuracy: 0.0579 - val_loss: 0.1400
Epoch 6/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 6s 149ms/step - accuracy: 0.0478 - loss: 0.1102 - val_accuracy: 0.0579 - val_loss: 0.1401
Epoch 7/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 5s 132ms/step - accuracy: 0.0485 - loss: 0.1101 - val_accuracy: 0.0579 - val_loss: 0.1402
Epoch 8/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 6s 148ms/step - accuracy: 0.0497 - loss: 0.1100 - val_accuracy: 0

c:\Users\LJM\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\LJM\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\LJM\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()